# Data Preprocessing – Master × Aux Join Pipeline

This notebook orchestrates the full preprocessing pipeline:

| Step | What happens |
|------|-------------|
| 1 | Load configuration from `config.json` |
| 2 | Load `master_dataset_car.parquet` |
| 3 | Load `car_with_dep_factor_fold.parquet` |
| 4 | Inner-join on `vin_date` and report dropped rows |
| 5 | Run data-quality check (NaN & zeros per column) |
| 6 | Save combined dataset as Parquet |
| 7 | Save quality report as CSV |

All business logic lives in `code/data_processing.py`.

---
## 1 · Environment setup

In [ ]:
import os
import sys

# Project root is the directory that contains this notebook
PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.json")

print(f"Project root : {PROJECT_ROOT}")
print(f"Config path  : {CONFIG_PATH}")

---
## 2 · Load configuration

In [ ]:
from data_processing import load_config

config = load_config(CONFIG_PATH)

# ── Global settings ──────────────────────────────────────────────────────────
BASE_DIR    = config["global"]["base_data_dir"]
MASTER_FILE = config["global"]["master_data"]
AUX_FILE    = config["global"]["aux_data_car_dep_fold"]
MODEL_YEAR  = config["global"]["model_year"]

# ── Preprocess settings ───────────────────────────────────────────────────────
JOIN_KEY      = config["preprocess"]["joinkey"]
COMBINED_OUT  = config["preprocess"]["combined_output"]
QUALITY_DIR   = config["preprocess"]["quality_report_dir"]
QUALITY_FILE  = config["preprocess"]["quality_report_file"]

print(f"Model year   : {MODEL_YEAR}")
print(f"Base dir     : {BASE_DIR}")
print(f"Master file  : {MASTER_FILE}")
print(f"Aux file     : {AUX_FILE}")
print(f"Join key     : {JOIN_KEY}")
print(f"Output file  : {COMBINED_OUT}")
print(f"Quality dir  : {QUALITY_DIR}")
print(f"Quality file : {QUALITY_FILE}")

---
## 3 · Load master dataset

In [ ]:
from data_processing import load_master_data

master_df = load_master_data(BASE_DIR, MASTER_FILE)

print(f"✅ master_data loaded  →  {master_df.shape[0]:,} rows × {master_df.shape[1]} columns")
master_df.head()

In [ ]:
print("master_data dtypes:")
print(master_df.dtypes.to_string())
print()
print("master_data basic stats:")
master_df.describe(include='all')

---
## 4 · Load auxiliary dataset

In [ ]:
from data_processing import load_aux_data

aux_df = load_aux_data(BASE_DIR, AUX_FILE)

print(f"✅ aux_data loaded  →  {aux_df.shape[0]:,} rows × {aux_df.shape[1]} columns")
aux_df.head()

In [ ]:
print("aux_data dtypes:")
print(aux_df.dtypes.to_string())
print()
print("aux_data basic stats:")
aux_df.describe(include='all')

---
## 5 · Inner join on `vin_date` + row-drop report

In [ ]:
from data_processing import join_data
import pandas as pd

combined_df, drop_report = join_data(master_df, aux_df, JOIN_KEY, join_type="inner")

# ── Pretty-print the drop report ─────────────────────────────────────────────
drop_summary = pd.DataFrame(
    [
        {"Metric": "Master rows (before join)",       "Value": f"{drop_report['master_rows']:,}"},
        {"Metric": "Aux rows (before join)",          "Value": f"{drop_report['aux_rows']:,}"},
        {"Metric": "Combined rows (after join)",      "Value": f"{drop_report['combined_rows']:,}"},
        {"Metric": "Common join keys",                "Value": f"{drop_report['common_keys']:,}"},
        {"Metric": "Keys only in master",             "Value": f"{drop_report['keys_only_in_master']:,}"},
        {"Metric": "Keys only in aux",                "Value": f"{drop_report['keys_only_in_aux']:,}"},
        {"Metric": "Rows dropped from master",        "Value": f"{drop_report['dropped_from_master']:,}  ({drop_report['pct_dropped_master']:.4f}%)"},
        {"Metric": "Rows dropped from aux",           "Value": f"{drop_report['dropped_from_aux']:,}  ({drop_report['pct_dropped_aux']:.4f}%)"},
    ]
)

if drop_report['dropped_from_master'] == 0 and drop_report['dropped_from_aux'] == 0:
    print("✅ PERFECT JOIN – no rows dropped!")
else:
    print("⚠️  Some rows were dropped during the inner join – see report below.")

print()
display(drop_summary.set_index("Metric"))

In [ ]:
print(f"Combined dataset shape: {combined_df.shape[0]:,} rows × {combined_df.shape[1]} columns")
print(f"Columns: {list(combined_df.columns)}")
print()
combined_df.head()

---
## 6 · Data quality report

In [ ]:
from data_processing import check_data_quality

quality_df = check_data_quality(combined_df)

# ── Summary flags ─────────────────────────────────────────────────────────────
total_cols   = len(quality_df)
cols_with_nan  = int((quality_df["nan_count"] > 0).sum())
cols_with_zero = int((quality_df["zero_count"].fillna(0) > 0).sum())

print(f"Total columns    : {total_cols}")
print(f"Columns with NaN : {cols_with_nan}")
print(f"Columns with 0s  : {cols_with_zero}")
print()

print("── Columns WITH NaN values ────────────────────")
display(quality_df[quality_df["nan_count"] > 0])

print("── Numeric columns WITH zero values ───────────")
display(quality_df[quality_df["zero_count"].notna() & (quality_df["zero_count"] > 0)])

print("── Full quality report ─────────────────────────")
display(quality_df)

---
## 7 · Save combined dataset

In [ ]:
from data_processing import save_combined_data

combined_path = save_combined_data(combined_df, BASE_DIR, COMBINED_OUT)

print(f"✅ Combined dataset saved to:")
print(f"   {combined_path}")

---
## 8 · Save quality report

In [ ]:
from data_processing import save_quality_report

report_path = save_quality_report(quality_df, BASE_DIR, QUALITY_DIR, QUALITY_FILE)

print(f"✅ Quality report saved to:")
print(f"   {report_path}")

---
## 9 · Pipeline summary

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║              PREPROCESSING PIPELINE COMPLETE                 ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Model year      : {MODEL_YEAR}                                       ║")
print(f"║  Master rows     : {drop_report['master_rows']:>10,}                              ║")
print(f"║  Aux rows        : {drop_report['aux_rows']:>10,}                              ║")
print(f"║  Combined rows   : {drop_report['combined_rows']:>10,}                              ║")
print(f"║  Dropped (master): {drop_report['dropped_from_master']:>10,}  ({drop_report['pct_dropped_master']:.4f}%)              ║")
print(f"║  Dropped (aux)   : {drop_report['dropped_from_aux']:>10,}  ({drop_report['pct_dropped_aux']:.4f}%)              ║")
print(f"║  Cols w/ NaN     : {cols_with_nan:>10}                              ║")
print(f"║  Cols w/ zeros   : {cols_with_zero:>10}                              ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Combined saved  → {COMBINED_OUT}")
print(f"║  Quality report  → {QUALITY_DIR}/{QUALITY_FILE}")
print("╚══════════════════════════════════════════════════════════════╝")